# 04: Trend Fusion Engine

**Objective**: Merge the structured sell-through scorecard with
the unstructured social signal scorecard to generate the **Trend
Intelligence Master** the core dataset that answers:

> *"Which styles should a D2C fashion brand over-index on for their next production run?"*

### Scoring Formula

| Signal | Weight | Source |
|--------|--------|--------|
| Sell-through velocity | 0.25 | H&M transactions |
| Sentiment lift | 0.25 | Fine-tuned RoBERTa |
| Trend momentum | 0.20 | Google Trends |
| Style buzz | 0.15 | BART zero-shot |
| Cluster strength | 0.15 | MiniLM + HDBSCAN |

### Quadrant Classification

| Quadrant | Social Signal | Sell-Through | Action |
|----------|--------------|--------------|--------|
| **HOT** | High | High | Over-index aggressively |
| **EMERGING** | High | Low | Test & invest |
| **FADING** | Low | High | Maintain, do not grow |
| **COLD** | Low | Low | Deprioritize |

---

## 4.1 Setup

In [1]:
import sys, os, logging, warnings
warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/FashionTrendAnalyzer"
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import trend_scoring_config, CATEGORY_TAXONOMY
from src.data_loader import load_processed, save_processed
from src.trend_scorer import (
    build_trend_intelligence_master,
    generate_top_n_recommendations,
    map_hm_product_to_category,
)
from src.viz import trend_quadrant_scatter, methodology_waterfall

print("Setup complete.")
print(f"\nScoring weights: {trend_scoring_config.model_dump()}")

Mounted at /content/drive
Setup complete.

Scoring weights: {'w_sell_through': 0.25, 'w_sentiment': 0.25, 'w_trend_momentum': 0.2, 'w_style_buzz': 0.15, 'w_cluster_strength': 0.15, 'quadrant_threshold': 0.5}


## 4.2 Load Both Scorecards

In [2]:
sell_through = load_processed("sell_through_scorecard")
social_signals = load_processed("social_signal_scorecard")

print(f"Sell-through scorecard: {sell_through.shape}")
print(f"Social signal scorecard: {social_signals.shape}")
print(f"\nSell-through columns: {list(sell_through.columns)}")
print(f"Social signal columns: {list(social_signals.columns)}")

Sell-through scorecard: (2634, 17)
Social signal scorecard: (10, 22)

Sell-through columns: ['product_type_name', 'colour_group_name', 'total_units', 'total_revenue', 'weeks_active', 'avg_weekly_units', 'median_weekly_units', 'velocity_score', 'revenue_per_sku', 'wow_growth_rate', 'first_half_units', 'second_half_units', 'velocity_score_normalized', 'total_revenue_normalized', 'revenue_per_sku_normalized', 'sell_through_composite', 'sell_through_class']
Social signal columns: ['category', 'mean_sentiment', 'median_sentiment', 'std_sentiment', 'review_count', 'positive_ratio', 'sentiment_normalized', 'dominant_style', 'dominant_proportion', 'style_diversity', 'style_buzz_normalized', 'cluster', 'cluster_theme', 'density', 'cluster_strength_normalized', 'avg_momentum', 'max_momentum', 'avg_interest', 'breakout_count', 'keyword_count', 'breakout_flag', 'trend_momentum_normalized']


## 4.3 Taxonomy Mapping

The sell-through data uses H&M's product taxonomy (`product_type_name`)
while social signals use our unified category taxonomy. We bridge them here.

In [3]:
# Map H&M product types to our taxonomy
sell_through["category"] = sell_through["product_type_name"].apply(map_hm_product_to_category)

print("H&M products mapped to categories:")
print(sell_through["category"].value_counts())

print(f"\nSocial signal categories:")
print(social_signals["category"].value_counts())

common = set(sell_through["category"]) & set(social_signals["category"])
print(f"\nCommon categories for fusion: {len(common)} — {sorted(common)}")

H&M products mapped to categories:
category
Other          1370
Tops            419
Accessories     266
Trousers        149
Jackets         147
Shoes           146
Dresses          50
Skirts           47
Knitwear         40
Name: count, dtype: int64

Social signal categories:
category
Accessories          1
Dresses              1
Jackets              1
Knitwear             1
Other                1
Prints & Patterns    1
Shorts               1
Skirts               1
Tops                 1
Trousers             1
Name: count, dtype: int64

Common categories for fusion: 8 — ['Accessories', 'Dresses', 'Jackets', 'Knitwear', 'Other', 'Skirts', 'Tops', 'Trousers']


## 4.4 Build Trend Intelligence Master

In [4]:
master = build_trend_intelligence_master(
    sell_through=sell_through,
    social_signals=social_signals,
)

print(f"Trend Intelligence Master: {master.shape}")
print(f"\nQuadrant distribution:")
print(master["quadrant"].value_counts())

master.sort_values("trend_intelligence_score", ascending=False)

Trend Intelligence Master: (11, 32)

Quadrant distribution:
quadrant
EMERGING    4
FADING      4
HOT         2
COLD        1
Name: count, dtype: int64


,category,sell_through_composite,velocity_score_normalized,total_revenue,total_units,wow_growth_rate,n_style_color_combos,mean_sentiment,median_sentiment,std_sentiment,...,max_momentum,avg_interest,breakout_count,keyword_count,breakout_flag,trend_momentum_normalized,trend_intelligence_score,social_composite,quadrant,recommendation
4,Other,0.089885,0.003496,13379.567831,639833.0,-0.028995,1370.0,0.995033,0.997494,0.004982,...,NaN,NaN,NaN,NaN,None,0.500000,0.559521,0.716066,EMERGING,"Test & invest — high social buzz, sales haven'..."
7,Shorts,0.500000,NaN,NaN,NaN,NaN,NaN,0.695691,0.994736,0.547911,...,5.853147,88.000000,1.0,1.0,True,1.000000,0.494673,0.492897,HOT,Over-index aggressively — strong demand + risi...
5,Prints & Patterns,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.587413,14.666667,0.0,3.0,False,0.000000,0.400000,0.366667,HOT,Over-index aggressively — strong demand + risi...
6,Shoes,0.092077,0.002662,2087.733881,49973.0,-0.086264,146.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.500000,0.398019,0.500000,EMERGING,"Test & invest — high social buzz, sales haven'..."
8,Skirts,0.062588,0.013361,3333.605441,93901.0,-0.481351,47.0,0.694816,0.993132,0.544290,...,2.583916,70.000000,1.0,1.0,True,0.423314,0.387342,0.495593,EMERGING,"Test & invest — high social buzz, sales haven'..."
0,Accessories,0.085333,0.002848,1858.775220,105218.0,-0.109851,266.0,NaN,NaN,NaN,...,2.318182,30.000000,1.0,2.0,True,0.219161,0.340166,0.425110,EMERGING,"Test & invest — high social buzz, sales haven'..."
10,Trousers,0.101413,0.027665,19840.813390,610910.0,0.093935,149.0,0.713261,0.994596,0.527000,...,3.027972,86.000000,2.0,2.0,True,0.415913,0.248889,0.298048,FADING,Maintain current levels — selling well but soc...
3,Knitwear,0.135731,0.006768,1156.292288,38026.0,1.920358,40.0,0.663790,0.993525,0.575040,...,NaN,NaN,NaN,NaN,None,0.500000,0.246926,0.283991,FADING,Maintain current levels — selling well but soc...
2,Jackets,0.113834,0.005652,7721.191915,118009.0,0.178539,147.0,0.678107,0.994913,0.579916,...,2.702797,56.500000,2.0,2.0,True,0.266961,0.183685,0.206969,FADING,Maintain current levels — selling well but soc...
9,Tops,0.099528,0.019368,27192.810678,1197885.0,0.515784,419.0,0.660314,0.992303,0.574646,...,3.972028,64.600000,2.0,5.0,True,0.325863,0.171390,0.195343,FADING,Maintain current levels — selling well but soc...


In [5]:
# Signal correlation matrix
signal_cols = [
    "sell_through_composite", "sentiment_normalized",
    "trend_momentum_normalized", "style_buzz_normalized",
    "cluster_strength_normalized", "trend_intelligence_score",
]
available_cols = [c for c in signal_cols if c in master.columns]

fig = px.imshow(
    master[available_cols].corr(),
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Signal Correlation Matrix",
    template="plotly_white",
    text_auto=".2f",
)
fig.update_layout(height=500, width=600)
fig.show()

## 4.5 Trend Quadrant Visualization

In [6]:
# Fill revenue for sizing (use 1 if missing)
if "total_revenue" not in master.columns or master["total_revenue"].isna().all():
    master["total_revenue"] = 1
master["total_revenue"] = master["total_revenue"].fillna(1).clip(lower=1)

fig = trend_quadrant_scatter(master)
fig.show()

## 4.6 Methodology Breakdown

For each category, decompose the trend intelligence score into its component
signal contributions.

In [7]:
# Show waterfall for the top-scoring category
top_category = master.iloc[0]

scores = {
    "Sell-Through": top_category.get("sell_through_composite", 0.5),
    "Sentiment": top_category.get("sentiment_normalized", 0.5),
    "Trend Momentum": top_category.get("trend_momentum_normalized", 0.5),
    "Style Buzz": top_category.get("style_buzz_normalized", 0.5),
    "Cluster Strength": top_category.get("cluster_strength_normalized", 0.5),
}
weights = {
    "Sell-Through": trend_scoring_config.w_sell_through,
    "Sentiment": trend_scoring_config.w_sentiment,
    "Trend Momentum": trend_scoring_config.w_trend_momentum,
    "Style Buzz": trend_scoring_config.w_style_buzz,
    "Cluster Strength": trend_scoring_config.w_cluster_strength,
}

fig = methodology_waterfall(top_category["category"], scores, weights)
fig.show()

print(f"\nCategory: {top_category['category']}")
print(f"Final Score: {top_category['trend_intelligence_score']:.4f}")
print(f"Quadrant: {top_category['quadrant']}")


Category: Other
Final Score: 0.5595
Quadrant: EMERGING


## 4.7 Top N Recommendations

In [8]:
recs = generate_top_n_recommendations(master, top_n=10)
recs

,category,quadrant,trend_intelligence_score,sell_through_composite,social_composite,dominant_style,recommendation,rank,confidence
4,Other,EMERGING,0.559521,0.089885,0.716066,oversized comfort,"Test & invest — high social buzz, sales haven'...",1,56.0
7,Shorts,HOT,0.494673,0.500000,0.492897,quiet luxury,Over-index aggressively — strong demand + risi...,2,49.5
5,Prints & Patterns,HOT,0.400000,0.500000,0.366667,None,Over-index aggressively — strong demand + risi...,3,40.0
6,Shoes,EMERGING,0.398019,0.092077,0.500000,NaN,"Test & invest — high social buzz, sales haven'...",4,39.8
8,Skirts,EMERGING,0.387342,0.062588,0.495593,quiet luxury,"Test & invest — high social buzz, sales haven'...",5,38.7
0,Accessories,EMERGING,0.340166,0.085333,0.425110,None,"Test & invest — high social buzz, sales haven'...",6,34.0
10,Trousers,FADING,0.248889,0.101413,0.298048,quiet luxury,Maintain current levels — selling well but soc...,7,24.9
3,Knitwear,FADING,0.246926,0.135731,0.283991,oversized comfort,Maintain current levels — selling well but soc...,8,24.7
2,Jackets,FADING,0.183685,0.113834,0.206969,oversized comfort,Maintain current levels — selling well but soc...,9,18.4
9,Tops,FADING,0.171390,0.099528,0.195343,quiet luxury,Maintain current levels — selling well but soc...,10,17.1


## 4.8 Save Master Dataset

In [9]:
save_processed(master, "trend_intelligence_master")
save_processed(recs, "top_recommendations")

print(f"Saved trend_intelligence_master.parquet — {len(master)} categories")
print(f"Saved top_recommendations.parquet — {len(recs)} recommendations")

Saved trend_intelligence_master.parquet — 11 categories
Saved top_recommendations.parquet — 10 recommendations
